In [1]:
import json
import requests
import os
import time
import getpass
from huggingface_hub import notebook_login

from datasets import load_dataset
from fireworks.client import Fireworks
from pydantic import BaseModel, Field
from transformers import AutoTokenizer
import pandas as pd


In [2]:
os.environ["FIREWORKS_API_KEY"] = getpass.getpass("fireworks api:")

In [3]:
notebook_login()

In [7]:
client = Fireworks(api_key=os.environ["FIREWORKS_API_KEY"])

model_id = "meta-llama/Meta-Llama-3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

/home/markt/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [23]:
sample_data = pd.read_csv('english-danish-openai.csv')
sample_data['English'][1]

"She baked a delicious chocolate cake for her friend's birthday."

In [32]:

def create_eng_to_danish_json_prompt(english_language, danish_language):
    return {"messages": [
        {"role": "system", "content": "Translate English sentence to Danish"},
        {"role": "user", "content": english_language},
        {"role": "assistant", "content": danish_language}
    ]}

def create_json_language_list_by_csv(language_list_csv):
    language_prompt_list = list()
    language_csv = pd.read_csv(language_list_csv)
    for i, row in enumerate(language_csv.iterrows()):
        prompt_msg = create_eng_to_danish_json_prompt(row[1]['English'], row[1]['Danish'])
        language_prompt_list.append(prompt_msg)
    return language_prompt_list

lang_json_list = create_json_language_list_by_csv('english-danish-openai.csv')

lang_json_list


[{'messages': [{'role': 'system',
    'content': 'Translate English sentence to Danish'},
   {'role': 'user',
    'content': 'The cat sat on the windowsill  watching the birds outside.'},
   {'role': 'assistant',
    'content': 'Katten sad på vindueskarmen og kiggede på fuglene udenfor.'}]},
 {'messages': [{'role': 'system',
    'content': 'Translate English sentence to Danish'},
   {'role': 'user',
    'content': "She baked a delicious chocolate cake for her friend's birthday."},
   {'role': 'assistant',
    'content': 'Hun bagte en lækker chokoladekage til sin vens fødselsdag.'}]},
 {'messages': [{'role': 'system',
    'content': 'Translate English sentence to Danish'},
   {'role': 'user',
    'content': 'The new park in the city center has become a popular spot for families.'},
   {'role': 'assistant',
    'content': 'Den nye park i byens centrum er blevet et populært sted for familier.'}]},
 {'messages': [{'role': 'system',
    'content': 'Translate English sentence to Danish'},
  

In [33]:
def write_dataset_to_json_l(dataset_name, json_language_dataset):
    with open(f'{dataset_name}.jsonl', 'w') as f:
        for obj in json_language_dataset:
            json.dump(obj, f)
            f.write('\n')

write_dataset_to_json_l('danish_to_english_dataset', lang_json_list)

In [35]:
dataset_file_name = 'danish_to_english_dataset.jsonl'
dataset_id = 'danish-to-english-dataset-v1'

!firectl create dataset {dataset_id} {dataset_file_name}

2024/07/05 19:45:10 There are updates available.
Current version: 1.1.2
Latest version: 1.2.0

To upgrade to the latest version, run
  $ sudo firectl upgrade

31.88 KiB / 31.88 KiB [---------------------------] 100.00% 125.05 KiB p/s 500ms


In [43]:
!firectl version

1.2.0


In [44]:
!firectl get dataset {dataset_id}

Name: accounts/m44rkt1-d483d1/datasets/danish-to-english-dataset-v1
Display Name: 
Create Time: 2024-07-05 19:45:11
State: READY
Status: OK
Example Count: 119


In [54]:
!firectl create fine-tuning-job --settings-file finetuning_danish_to_english_v1.yaml --display-name danish-english-v2 --dataset {dataset_id}

Name: accounts/m44rkt1-d483d1/fineTuningJobs/83ba6bb9b2ec4ca48bad7e473b87a113
Display Name: danish-english-v2
Create Time: 2024-07-05 21:28:52
State: CREATING
Dataset: accounts/m44rkt1-d483d1/datasets/danish-to-english-dataset-v1
Created By: m44rkt1@gmail.com
Container Version: 
Model Id: 
Wandb Url: https://wandb.ai/markat1/danish-english-fine-tuning-workshop-1/groups/group-83ba6bb9b2ec4ca48bad7e473b87a113/workspace
Conversation:
  Jinja Template: {%- set _mode = mode | default('generate', true) -%}
{%- set stop_token = '<|eot_id|>' -%}
{%- set message_roles = ['SYSTEM', 'USER', 'ASSISTANT'] -%}
{%- set ns = namespace(initial_system_message_handled=false, last_assistant_index_for_eos=-1, messages=messages) -%}
{%- for message in ns.messages -%}
    {%- if not message.get('role') -%}
        {{ raise_exception('Key [role] is missing. Original input: ' +  message|tojson) }}
    {%- endif -%}
    {%- if message['role'] | upper not in message_roles -%}
        {{ raise_exception('Invalid 

In [57]:

lang_tuning_job_id = '83ba6bb9b2ec4ca48bad7e473b87a113'

!firectl get fine-tuning-job {lang_tuning_job_id}

Name: accounts/m44rkt1-d483d1/fineTuningJobs/83ba6bb9b2ec4ca48bad7e473b87a113
Display Name: danish-english-v2
Create Time: 2024-07-05 21:28:52
State: COMPLETED
Dataset: accounts/m44rkt1-d483d1/datasets/danish-to-english-dataset-v1
Status: OK
Created By: m44rkt1@gmail.com
Container Version: 
Model Id: 
Wandb Url: https://wandb.ai/markat1/danish-english-fine-tuning-workshop-1/groups/group-83ba6bb9b2ec4ca48bad7e473b87a113/workspace
Conversation:
  Jinja Template: {%- set _mode = mode | default('generate', true) -%}
{%- set stop_token = '<|eot_id|>' -%}
{%- set message_roles = ['SYSTEM', 'USER', 'ASSISTANT'] -%}
{%- set ns = namespace(initial_system_message_handled=false, last_assistant_index_for_eos=-1, messages=messages) -%}
{%- for message in ns.messages -%}
    {%- if not message.get('role') -%}
        {{ raise_exception('Key [role] is missing. Original input: ' +  message|tojson) }}
    {%- endif -%}
    {%- if message['role'] | upper not in message_roles -%}
        {{ raise_excepti

In [58]:
!firectl deploy {lang_tuning_job_id}

In [59]:
!firectl list models

NAME                              CREATE TIME          KIND           CHAT  PUBLIC  STATE  STATUS MESSAGE
755fd9d02a7046ef8368d4a15c86f3cc  2024-07-05 20:10:19  HF_PEFT_ADDON  true  false   READY  
83ba6bb9b2ec4ca48bad7e473b87a113  2024-07-05 21:33:24  HF_PEFT_ADDON  true  false   READY  
bfa866cd4b4841669796a0deee36d771  2024-06-27 21:36:41  HF_PEFT_ADDON  true  false   READY  
c2c7013101774cd19f0a18cc2f109a29  2024-06-27 21:08:19  HF_PEFT_ADDON  true  false   READY  

Total size: 4


In [61]:
ft_model_name = f'accounts/m44rkt1-d483d1/models/{lang_tuning_job_id}'
base_model_name = "accounts/fireworks/models/llama-v3-8b-instruct"

sample_data = pd.read_csv('english-danish-openai.csv')

def generate_translations(model, english_sentences):
    responses = list()
    for i, sentence in enumerate(english_sentences):
        msg = [
            {"role": "system", "content": 'Translate the English sentence to Danish. Your response must contain ONLY the translated sentence.'},
            {"role": "user", "content": sentence},
        ]
        response = client.chat.completions.create(
            model=model,
            messages=msg,
            temperature=0,
        )

        response = response.choices[0].message.content
        print(response)
        responses.append(response)
    return responses


generate_translations(ft_model_name , sample_data['English'].tolist())


<|start_header_id|>Katten sad på vindueskarmen og kiggede på fuglene udenfor.
<|start_header_id|>Hun bagte en lækker chokoladekage til sin vens fødsel.
<|start_header_id|>Den nye park i byens centrum er blevet et populært sted for familier.
<|start_header_id|>Han indførte hurtigt, at lære et nyt sprog kræver tålmodighed og praksis.
<|start_header_id|>```
<|start_header_id|>De besluttede at tilbringe deres ferie med at udforske de fjerne øer i Stillehavet.
<|start_header_id|>Forskeren gjorde en banebrydende opdagelse, der kunne ændre medicinens fremtid.
<|start_header_id|><|start_header_id|>Børnene var begejstrede for at starte deres første skoledag efter somferien.
<|start_header_id|><|start_header_id|>Hun gennemførte maratonen trods de udfordrende vejrforhold.
<|start_header_id|>Den innovative startup har til formål at skabe bæredygtige løsninger for byliv.
<|start_header_id|>Hans bedstefar fortalte ham historier om gamle dage, da landsbyen var meget mindre.
<|start_header_id|>Duften 

['<|start_header_id|>Katten sad på vindueskarmen og kiggede på fuglene udenfor.',
 '<|start_header_id|>Hun bagte en lækker chokoladekage til sin vens fødsel.',
 '<|start_header_id|>Den nye park i byens centrum er blevet et populært sted for familier.',
 '<|start_header_id|>Han indførte hurtigt, at lære et nyt sprog kræver tålmodighed og praksis.',
 '<|start_header_id|>```',
 '<|start_header_id|>De besluttede at tilbringe deres ferie med at udforske de fjerne øer i Stillehavet.',
 '<|start_header_id|>Forskeren gjorde en banebrydende opdagelse, der kunne ændre medicinens fremtid.',
 '<|start_header_id|><|start_header_id|>Børnene var begejstrede for at starte deres første skoledag efter somferien.',
 '<|start_header_id|><|start_header_id|>Hun gennemførte maratonen trods de udfordrende vejrforhold.',
 '<|start_header_id|>Den innovative startup har til formål at skabe bæredygtige løsninger for byliv.',
 '<|start_header_id|>Hans bedstefar fortalte ham historier om gamle dage, da landsbyen va